In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import time
import json
import os

### загрузка конфига

In [ ]:
def load_config(config_path='../config.json'):
    with open(config_path, 'r', encoding='utf-8') as f:
        return json.load(f)

config = load_config()
params = config['parameters']
lms_cfg = params['landmarks']    

### инициализация medapipe

In [ ]:
mp_face_mesh = mp.solutions.face_mesh

### функции для анализа

In [ ]:
def get_head_pose_v2(face_landmarks, img_w, img_h):
    """Вычисляет Pitch и Yaw головы через PnP."""
    face_2d = []
    model_3d = np.array([
        [0.0, 0.0, 0.0],          # Нос
        [0.0, -330.0, -65.0],    # Подбородок
        [-225.0, 170.0, -135.0], # Левый глаз
        [225.0, 170.0, -135.0],  # Правый глаз
        [-150.0, -150.0, -125.0],# Левый рот
        [150.0, -150.0, -125.0]  # Правый рот
    ], dtype=np.float64)

    for idx in lms_cfg['pnp_indices']:
        lm = face_landmarks.landmark[idx]
        face_2d.append([lm.x * img_w, lm.y * img_h])

    face_2d = np.array(face_2d, dtype=np.float64)
    focal_length = img_w
    cam_matrix = np.array([[focal_length, 0, img_w / 2],
                           [0, focal_length, img_h / 2],
                           [0, 0, 1]])

    success, rot_vec, trans_vec = cv2.solvePnP(model_3d, face_2d, cam_matrix, np.zeros((4, 1)))
    rmat, _ = cv2.Rodrigues(rot_vec)

    pitch = np.arcsin(-rmat[2, 0])
    yaw = np.arctan2(rmat[2, 1], rmat[2, 2])
    return np.degrees(pitch), np.degrees(yaw)

In [ ]:
def get_gaze_ratio(face_landmarks, eye_indices, iris_indices):
    """Вычисляет относительное положение зрачка (0.0 - 1.0)."""
    lm = face_landmarks.landmark
    x_left = lm[eye_indices[0]].x
    x_right = lm[eye_indices[8]].x
    x_iris = np.mean([lm[i].x for i in iris_indices])
    
    width = abs(x_right - x_left)
    if width == 0: 
        return 0.5
    return (x_iris - min(x_left, x_right)) / width

In [ ]:
def analyze_gaze_advanced(face_lms, pitch, raw_yaw, params):
    """Основная логика Advanced алгоритма с подтягиванием параметров из JSON."""
    # 1. Нормализация Yaw
    yaw = raw_yaw - params['yaw_center_offset'] if raw_yaw > 0 else raw_yaw + params['yaw_center_offset']
    
    # 2. Текущее положение зрачков
    ratio_l = get_gaze_ratio(face_lms, lms_cfg['left_eye_indices'], lms_cfg['left_iris_indices'])
    ratio_r = get_gaze_ratio(face_lms, lms_cfg['right_eye_indices'], lms_cfg['right_iris_indices'])
    avg_gaze = (ratio_l + ratio_r) / 2

    # 3. Базовая отбраковка (если затылок или слишком сильный наклон)
    yaw_lim = params['head_pose_thresholds']['yaw_limit']
    p_min, p_max = params['head_pose_thresholds']['pitch_limit']
    if not (p_min < pitch < p_max) or abs(yaw) > yaw_lim:
        return False

    # 4. Динамический расчет идеального взгляда в камеру
    base_gaze = params['iris_tracking']['base_gaze']
    coef = params['iris_tracking']['yaw_gaze_coefficient']
    tolerance = params['iris_tracking']['gaze_tolerance']
    
    expected_gaze = base_gaze - (yaw * coef)
    
    # 5. Проверяем, насколько реальный взгляд совпадает с идеальным
    if abs(avg_gaze - expected_gaze) < tolerance:
        return True
        
    return False


### главный цикл

In [ ]:
def process_video_prototype(video_path, config):
    if not os.path.exists(video_path):
        return None

    params = config['parameters']
    sampling_rate = config['processing']['sampling_rate']
    cap = cv2.VideoCapture(video_path)
    
    raw_data = []
    
    start_time = time.time()
    with mp_face_mesh.FaceMesh(refine_landmarks=True, min_detection_confidence=0.5) as face_mesh:
        frame_idx = 0
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break
                
            frame_idx += 1
            if frame_idx % sampling_rate != 0: continue
            
            # --- Замер яркости кадра ---
            gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            brightness = np.mean(gray_frame)
            
            h, w, _ = frame.shape
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = face_mesh.process(rgb_frame)

            if results.multi_face_landmarks:
                face_lms = results.multi_face_landmarks[0]
                pitch, raw_yaw = get_head_pose_v2(face_lms, w, h)
                
                yaw = raw_yaw - params['yaw_center_offset'] if raw_yaw > 0 else raw_yaw + params['yaw_center_offset']
                
                ratio_l = get_gaze_ratio(face_lms, lms_cfg['left_eye_indices'], lms_cfg['left_iris_indices'])
                ratio_r = get_gaze_ratio(face_lms, lms_cfg['right_eye_indices'], lms_cfg['right_iris_indices'])
                avg_gaze = (ratio_l + ratio_r) / 2
                
                raw_data.append({
                    'pitch': pitch, 
                    'yaw': yaw, 
                    'gaze': avg_gaze,
                    'brightness': brightness
                })
    cap.release()
    
    if not raw_data:
        return None
        
    median_gaze = np.median([d['gaze'] for d in raw_data])
    
    hits = 0
    rejected_by_pose = 0
    rejected_by_gaze = 0
    
    tolerance = params['iris_tracking']['gaze_tolerance']
    coef = params['iris_tracking']['yaw_gaze_coefficient']
    yaw_lim = params['head_pose_thresholds']['yaw_limit']
    p_min, p_max = params['head_pose_thresholds']['pitch_limit']
    
    DARK_THRESHOLD = 85.0 # Порог плохой освещенности (подбирается опытным путем)
    
    for d in raw_data:
        # 1. Проверка позы головы (работает всегда)
        if not (p_min < d['pitch'] < p_max) or abs(d['yaw']) > yaw_lim:
            rejected_by_pose += 1
            continue
            
        # 2. Гибридная проверка зрачка
        if d['brightness'] < DARK_THRESHOLD:
            # Если кадр темный, и поза головы прошла проверку выше - засчитываем успех!
            hits += 1
        else:
            # Если свет хороший, строго проверяем зрачки
            expected_gaze = median_gaze - (d['yaw'] * coef)
            if abs(d['gaze'] - expected_gaze) < tolerance:
                hits += 1
            else:
                rejected_by_gaze += 1

    score = hits / len(raw_data)
    
    print(f"\n--- Гибридный анализ для {os.path.basename(video_path)} ---")
    print(f"Всего кадров: {len(raw_data)}")
    print(f"Отбраковано из-за позы головы: {rejected_by_pose}")
    print(f"Отбраковано из-за зрачка (Светло): {rejected_by_gaze}")
    print(f"Успешных кадров (Hits): {hits}")
    print(f"Итоговый Score: {score:.4f}")
    
    return score


### запуск (change filename)

In [ ]:
video_to_test = "../media/expert_bad_light.mp4"
result = process_video_prototype(video_to_test, config)